# Fonction perturbée : pertes, erreur et classes de machines

Ce notebook accompagne l'exercice 7.1. La fonction exacte est connue : nous pouvons donc distinguer la perte empirique, calculée sur un nombre fini de données, de l'erreur fonctionnelle. Nous comparerons aussi la sensibilité des pertes quadratique, absolue et de Huber à quelques valeurs aberrantes (*'outliers'*).

## Parcours

1. [Données et perturbations](#donnees-regression)
2. [Une même machine, trois pertes](#pertes-regression)
3. [Plusieurs classes de machines](#machines-regression)
4. [Tube de la SVR](#tube-svr)
5. [Perte empirique et erreur fonctionnelle](#erreur-fonctionnelle)

In [1]:
import jax
import jax.numpy as jnp
import flax
from flax import nnx
import optax
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer
from sklearn.svm import SVR

jax.config.update("jax_enable_x64", True)
print(
    f"JAX {jax.__version__}, Flax {flax.__version__}, "
    f"Optax {optax.__version__}, scikit-learn {sklearn.__version__}"
)

JAX 0.11.1, Flax 0.12.9, Optax 0.2.8, scikit-learn 1.9.0


<a id="donnees-regression"></a>
## 1. Données et perturbations

Construisez des données pour $f_*(x)=\sin(2\pi x)$. À partir des mêmes observations faiblement perturbées, formez un second ensemble en déplaçant fortement trois valeurs. Cette expérience est construite ; elle ne prétend pas révéler une loi de probabilité sous-jacente.

In [ ]:
# À compléter.

<a id="pertes-regression"></a>
## 2. Une même machine, trois pertes

Fixez seize fonctions affines par morceaux en forme de chapeau et apprenez seulement leurs coefficients. La classe de machines est donc identique dans les trois expériences ; seule la fonction de perte change.

Comparez les ajustements obtenus avec

\[
\ell_2(r)=	frac12r^2,\qquad
\ell_1(r)=|r|,\qquad
\ell_{
m Huber}(r)=h_\delta(r).
\]

In [3]:
noeuds = jnp.linspace(0.0, 1.0, 16)
pas_noeuds = noeuds[1] - noeuds[0]


def base_p1(x):
    x = jnp.asarray(x).reshape(-1, 1)
    return jnp.maximum(1.0 - jnp.abs(x - noeuds.reshape(1, -1)) / pas_noeuds, 0.0)


def perte_residuelle(residu, nom, delta=0.15):
    if nom == "quadratique":
        return 0.5 * residu**2
    if nom == "absolue":
        return jnp.abs(residu)
    if nom == "Huber":
        valeur = jnp.abs(residu)
        return jnp.where(valeur <= delta, 0.5 * residu**2, delta * (valeur - 0.5 * delta))
    raise ValueError(nom)


def ajuster_p1(x, z, nom_perte, *, lamb=1e-4, alpha=0.03, iterations=2500):
    B = base_p1(x)
    z = jnp.asarray(z)

    def objectif(p):
        residu = B @ p - z
        regularisation = 0.5 * lamb * jnp.sum(jnp.diff(p) ** 2)
        return jnp.mean(perte_residuelle(residu, nom_perte)) + regularisation

    valeur_gradient = jax.jit(jax.value_and_grad(objectif))
    transformation = optax.adam(alpha)
    p = jnp.zeros(noeuds.size)
    etat = transformation.init(p)
    for _ in range(iterations):
        valeur, gradient = valeur_gradient(p)
        mises_a_jour, etat = transformation.update(gradient, etat, p)
        p = optax.apply_updates(p, mises_a_jour)
    return p, float(valeur)

In [ ]:
# À compléter.

<a id="machines-regression"></a>
## 3. Plusieurs classes de machines

Comparez maintenant quatre classes : polynômes, fonctions P1, SVR à noyau gaussien et MLP. Choisissez les hyperparamètres à partir de la validation, puis mesurez les erreurs sur une grille fine où $f_*$ est connue.

In [5]:
class MLP(nnx.Module):
    def __init__(self, largeur, *, rngs):
        self.W1 = nnx.Linear(1, largeur, rngs=rngs)
        self.W2 = nnx.Linear(largeur, largeur, rngs=rngs)
        self.W3 = nnx.Linear(largeur, 1, rngs=rngs)

    def __call__(self, x):
        x = nnx.tanh(self.W1(x))
        x = nnx.tanh(self.W2(x))
        return self.W3(x)


def norme_parametres(machine):
    return sum(jnp.sum(feuille**2) for feuille in jax.tree.leaves(nnx.state(machine, nnx.Param)))


def perte_mlp(machine, x, z, lamb=1e-5, delta=0.15):
    residu = machine(x).reshape(-1) - z
    valeur = jnp.abs(residu)
    huber = jnp.where(valeur <= delta, 0.5 * residu**2, delta * (valeur - 0.5 * delta))
    return jnp.mean(huber) + 0.5 * lamb * norme_parametres(machine)


@nnx.jit
def pas_mlp(machine, optimizer, x, z, lamb):
    valeur, gradient = nnx.value_and_grad(perte_mlp)(machine, x, z, lamb)
    optimizer.update(machine, gradient)
    return valeur


def entrainer_mlp(machine, x, z, *, lamb=1e-5, alpha=2e-3, iterations=2500):
    optimizer = nnx.Optimizer(machine, optax.adam(alpha), wrt=nnx.Param)
    x = jnp.asarray(x).reshape(-1, 1)
    z = jnp.asarray(z)
    for _ in range(iterations):
        valeur = pas_mlp(machine, optimizer, x, z, lamb)
    return float(valeur)


def predire_mlp(machine, x):
    return np.asarray(machine(jnp.asarray(x).reshape(-1, 1))).reshape(-1)

In [ ]:
# À compléter.

<a id="tube-svr"></a>
## 4. Tube de la SVR

Représentez le tube de demi-largeur $\epsilon$ et les vecteurs supports. Faites varier $\epsilon$ : comment évoluent le nombre de vecteurs supports et l'erreur fonctionnelle ?

In [ ]:
# À compléter.

<a id="erreur-fonctionnelle"></a>
## 5. Perte empirique et erreur fonctionnelle

Calculez pour chaque machine la RMSE et la MAE sur les données d'apprentissage et de validation, puis les erreurs $L^2$ et $L^\infty$ approchées sur la grille. Les quantités ne répondent pas à la même question.

In [ ]:
# À compléter.

## Bilan

La fonction de perte et la classe de machines sont deux choix distincts. La robustesse de $L^1$ ou de Huber concerne l'influence des résidus, tandis que la classe de machines gouverne les formes accessibles. La SVR ajoute une sélection d'exemples par son tube ; le MLP apprend lui-même ses représentations.